# Notebook 02 — A Knowledge Base for an AI SQL Assistant

**VidyaSankalp · Applied GenAI Engineering** · Tutor: Prudhvi Akella

This is the heart of the course. In Part 0 we stored five help articles and found them by meaning. Now we build
what a real AI assistant needs: a **knowledge base** that lets an LLM write correct SQL for a database it has never
seen.

```text
1  THE QUESTION        "What was our GMV last month?"  → the assistant must write SQL
2  WHAT THE LLM        which table? which column? what does GMV mean HERE? is there data?
   DOESN'T KNOW
3  WHY NOT PASTE       the whole knowledge ≈ 42,000 tokens — for every question
   EVERYTHING?            └─► store it, retrieve only the relevant pieces
4  THE KNOWLEDGE BASE  6 tabs, each answers one thing the LLM needs
5  THE DESIGN  ★       one row → one record: what to embed · what id · what metadata · what to skip
6  BUILD               337 records
7  LOAD                into Pinecone, namespace ecom-kb
8  RETRIEVE            the agent's three lookups: schema · examples · common
9  THE CONTEXT PACK ★  retrieved + followed + always in — what the LLM receives
10 KEEP IN SYNC        the workbook changes → the index follows
11 WHAT IT CAN'T DO    meaning, not structure
```

**Needs:** Part 0 done (same index `ecom-kb-tutorial`), the same `.env` (OpenAI + Pinecone keys), and
`kb_feed_ecom.xlsx` in this folder. **Time:** about 2½ hours.

## Chapter 1 — The question

An online store has a database with 14 tables: orders, order items, payments, shipments, customers, products,
suppliers and more. A manager types into an AI assistant:

> **"What was our GMV last month?"**

The assistant is an LLM. It must turn the question into SQL, run it, and answer. To do that it needs to know:

```text
"What was our GMV last month?"
        │
        ▼
   ┌─────────┐      SELECT ??? FROM ??? WHERE ???      ┌──────────┐
   │   LLM   │ ──────────────────────────────────────► │ database │ ──► answer
   └─────────┘                                          └──────────┘
        which table?  which column?  what is GMV?  which dates exist?
```

## Chapter 2 — What the LLM doesn't know

An LLM knows SQL and it knows what GMV usually means. It does **not** know **this** company's database. Left alone,
it guesses something like:

```sql
SELECT SUM(gmv) FROM sales WHERE order_date >= DATE_TRUNC('month', CURRENT_DATE - INTERVAL '1 month');
```

Wrong in four ways — and each fix is a fact that lives only in this company's documentation:

| The guess | The truth (from `kb_feed_ecom.xlsx`) | Where that fact lives |
|---|---|---|
| table `sales` | there is no `sales` table; orders are in `ecom.orders` | tab `nlq_table` |
| column `gmv` | no column is called GMV; it is `orders.total_amount` | tab `nlq_column` |
| GMV = everything paid | here GMV = **Revenue** = merchandise only — shipping, tax and discount **not** included | tab `glossary` |
| "last month" = the month before today | orders run **2023-01-01 to 2024-12-28**; nothing later exists | tab `capability_card` |

Plus two things that make the answer *good*, not just valid:

| Helpful fact | Where it lives |
|---|---|
| A proven query that already answers "revenue by month" | tab `nlq_examples` |
| How tables connect, when a question needs more than one | tab `nlq_join` |

> **So the LLM needs the right facts with every question.** Why not just give it all the facts, every time?

## Chapter 3 — Why not paste everything into the prompt?

Let's measure. First, setup — keys in `.env`, never in the notebook.

In [ ]:
%pip install -q openai pinecone python-dotenv openpyxl

In [ ]:
# ─────────────────────────────────────────────────────────────
# Setup: keys ──► OpenAI client + embed() ──► Pinecone index (same one as Part 0)
#
# Does NOT load any data yet.
# ─────────────────────────────────────────────────────────────
import os
import time

import openpyxl
from dotenv import load_dotenv
from openai import OpenAI
from pinecone import Pinecone, ServerlessSpec

load_dotenv()
for key in ("OPENAI_API_KEY", "PINECONE_API_KEY"):
    if not os.environ.get(key):
        raise ValueError(f"Please set {key} in your .env file.")

client = OpenAI()
EMBED_MODEL = "text-embedding-3-small"                 # the same model for records AND questions
INDEX_NAME, NAMESPACE = "ecom-kb-tutorial", "ecom-kb"


def embed(texts):
    """List of texts in → list of embeddings (1536 numbers each) out, same order."""
    response = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return [item.embedding for item in response.data]


pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
if not pc.has_index(INDEX_NAME):                       # in case Part 0 was skipped
    pc.create_index(name=INDEX_NAME, dimension=1536, metric="cosine",
                    spec=ServerlessSpec(cloud="aws", region="us-east-1"))
index = pc.Index(INDEX_NAME)
print("Connected to", INDEX_NAME)

In [ ]:
# ─────────────────────────────────────────────────────────────
# How big is the whole knowledge?
#
#   workbook ──► each tab ──► rows · characters ──► total ≈ tokens (about 4 characters per token)
#
# Does NOT embed anything. Just measures.
# ─────────────────────────────────────────────────────────────
workbook = openpyxl.load_workbook("kb_feed_ecom.xlsx", read_only=True)
KB_TABS = ["nlq_table", "nlq_column", "nlq_join", "glossary", "nlq_examples", "capability_card"]


def read_sheet(name):
    """One tab → list of rows; each row is {column name: value}. Row 1 holds the column names."""
    rows = list(workbook[name].iter_rows(values_only=True))
    header = rows[0]
    return [dict(zip(header, values)) for values in rows[1:] if not all(v is None for v in values)]


total_chars = 0
for tab in KB_TABS:
    rows = read_sheet(tab)
    chars = sum(len(" ".join(str(v) for v in row.values() if v is not None)) for row in rows)
    total_chars += chars
    print(f"{tab:16} {len(rows):>4} rows   {chars:>7,} characters")
print(f"\nTOTAL ≈ {total_chars:,} characters ≈ {total_chars // 4:,} tokens — for EVERY question")

About **42,000 tokens** of knowledge for a 14-table database. Pasting all of it into every prompt:

| Problem | Why it hurts |
|---|---|
| **Cost and speed** | every question pays for ~42,000 tokens of input, most of it irrelevant |
| **Noise** | the LLM must find 5 useful facts among 345 rows — more chances to pick the wrong column |
| **It doesn't scale** | a real company has hundreds or thousands of tables; it won't fit at all |

**The fix:** store the knowledge in a vector database, and for each question **retrieve only the few pieces that
matter**. The LLM gets a small, focused context. This pattern is called **RAG** — Retrieval-Augmented Generation.

```text
question ──► embed ──► Pinecone: closest knowledge records ──► a few facts ──► LLM writes SQL
                         (tables · columns · glossary ·
                          examples · capability card)
```

> **So what knowledge do we store, and how?**

## Chapter 4 — The knowledge base: six tabs

`kb_feed_ecom.xlsx` describes the database for the assistant. Each tab answers one thing the LLM needs to know:

| Tab | Rows | Answers the LLM's question | Example row |
|---|---|---|---|
| `nlq_table` | 14 | **What tables exist, and what is in each?** | `ecom.orders` — every order: who, when, how paid, status |
| `nlq_column` | 185 | **Which column holds this? What do people call it?** | `orders.total_amount` — merchandise subtotal; also called Revenue, Sales |
| `nlq_join` | 16 | **How do two tables connect?** | `orders` → `customers` on `customer_id`, many-to-one |
| `glossary` | 66 | **What does this business word mean here?** | **Revenue** = SUM(orders.total_amount); also called sales, GMV |
| `nlq_examples` | 61 | **Is there a proven query I can copy?** | "Which brands generate the most revenue?" + its SQL |
| `capability_card` | 3 | **What can — and can't — I answer?** | "orders after 2024-12-28 … nothing later" |

The workbook also has an `Instructions` tab — written for people who maintain the workbook, not for the assistant.
We don't load it.

```text
"What was our GMV last month?"   needs facts from:
   glossary         GMV → Revenue → merchandise only → orders.total_amount
   nlq_column       ecom.orders.total_amount (FLOAT)
   nlq_table        ecom.orders
   nlq_examples     "What is total merchandise revenue by month?" + SQL
   capability_card  data ends 2024-12-28
```

> **Each row will become one record in Pinecone. But what exactly goes into each record?**

## Chapter 5 — The design: one row → one record ★

From Part 0, a Pinecone record has three parts: **id + values (the embedding) + metadata**. For every tab we make
four decisions:

| Decision | Question to ask | Rule |
|---|---|---|
| **1 What to embed** | What will users' questions *look like*? | Embed text that **resembles the question** — descriptions, synonyms, example questions |
| **2 What id** | How will I find this record again to update or delete it? | A **stable name** built from the row's own key — never a random number |
| **3 What metadata** | What will I **filter** on? What must come **back** with a match? | Labels to filter (`sheet`, `table`) + what the LLM needs (the text, the SQL) |
| **4 What to skip** | Does this row or field carry meaning a search can use? | Skip what has no searchable meaning |

### Six walk-throughs — one real row from each tab

Each follows the "What was our GMV last month?" story: the table, the column, how it connects, what GMV means, a
proven query, and the limits.

#### 1 · `nlq_table` — what tables exist

```text
ROW (nlq_table)                                    RECORD
───────────────                                    ──────
table:             ecom.orders                     id:       table::ecom.orders
description:       # Orders                        EMBED:    "# Orders  **TABLE:** ecom.orders …
                   Every order placed on the        ──►        Every order placed on the online store
                   online store between 2023-01-01             between 2023-01-01 and 2024-12-28: who
                   and 2024-12-28: who ordered,                ordered, when, how they paid, where it
                   when, how they paid … (Markdown)            ships … (the whole description)"
grain:             One row per order               metadata: {sheet: nlq_table, table: ecom.orders, text}
row_count:         50000              ✗ not embedded
answers_questions: What is total revenue by month?; …  ✗ not embedded
```

The description is already a full Markdown write-up — grain, key measures, business meaning — so it is embedded as
it is. `row_count` is a number with no searchable meaning. `answers_questions` is kept out on purpose: Part 3 uses
those questions as the **test** — if they were embedded, the test would be an open-book exam.

#### 2 · `nlq_column` — which column holds it

```text
ROW (nlq_column)                                   RECORD
────────────────                                   ──────
table:            ecom.orders                      id:       column::ecom.orders.total_amount
column:           total_amount            ──►      EMBED:    "Column ecom.orders.total_amount (FLOAT).
data_type:        FLOAT                                       Merchandise subtotal = SUM(order_items.
description:      Merchandise subtotal …                      total_price); the amount paid. …
analyst_synonyms: Order Total; Revenue; …                     Also called: Order Total; Order Value;
sample_values:    55.92; 437.88; …   ✗ not embedded           Revenue; Sales; Subtotal."
                                                   metadata: {sheet: nlq_column, table: ecom.orders,
                                                              column: total_amount, data_type: FLOAT, text}
```

The analyst synonyms go **into** the text — users say "revenue", not "total_amount". `sample_values` are numbers:
"55.92" means nothing to a meaning search. `table` and `column` are metadata so a search can be narrowed to one table.

#### 3 · `nlq_join` — how two tables connect

```text
ROW (nlq_join)                                     RECORD
──────────────                                     ──────
left_table:  ecom.orders                           id:       join::ecom.orders->ecom.customers::customer_id
right_table: ecom.customers               ──►      EMBED:    "Join ecom.orders to ecom.customers on
join_on:     customer_id                                      customer_id (many-to-one). Each order
cardinality: many-to-one                                      belongs to one customer; a customer has
description: Each order belongs to one                        many orders (56 customers have none)."
             customer; a customer has many …       metadata: {sheet: nlq_join,
                                                              tables: [ecom.orders, ecom.customers], text}
```

A join row is five short fields; we **write them into one sentence** so it reads like the question it answers
("how do orders connect to customers?"). Both tables go into one metadata **list**, `tables`, so a filter on
either table finds this join.

#### 4 · `glossary` — what a business word means here

```text
ROW (glossary)                                     RECORD
──────────────                                     ──────
term:     Revenue                                  id:       glossary::Revenue
category: metric                        ──►        EMBED:    "Revenue (metric): Merchandise sales:
meaning:  Merchandise sales:                                  SUM(orders.total_amount) — shipping, tax
          SUM(orders.total_amount) — shipping,                and discount NOT included …
          tax and discount NOT included …                     Also called: sales; gross sales; GMV; …
synonyms: sales; gross sales; GMV; turnover …                 Columns: ecom.orders.total_amount; …"
maps_to:  ecom.orders.total_amount;                metadata: {sheet: glossary, term: Revenue,
          ecom.order_items.total_price                        category: metric, maps_to: [2 columns], text}
```

**Why the synonyms are embedded:** the word **GMV** appears in exactly one place in the whole workbook — this
row's synonyms. Embed them, and "What was our GMV…" lands right here. Leave them out, and the model has to guess
that GMV means Revenue. This is how we teach the assistant **company jargon**. `maps_to` is also kept as a metadata
**list**: chapter 9 follows it to fetch the exact column records.

#### 5 · `nlq_examples` — a proven query to copy

```text
ROW (nlq_examples)                                 RECORD
──────────────────                                 ──────
question:    What is total merchandise             id:       nlq_examples::1
             revenue by month?            ──►      EMBED:    "What is total merchandise revenue by
paraphrases: monthly revenue; sales per                       month? Also asked as: monthly revenue;
             month; revenue trend by month                    sales per month; revenue trend by month"
query:       SELECT date_trunc('month', …)         metadata: {sheet: nlq_examples, tables_used: [ecom.orders],
             … ✗ not embedded                                 pattern: aggregation,
tables_used: ecom.orders                                      sql: "SELECT date_trunc('month', …", text}
pattern:     aggregation
verified:    Y    owner, source, … ✗ not embedded
```

The **question and its paraphrases** are embedded — they look like what users ask. The **SQL is not embedded**:
nobody searches with SQL. It rides along in metadata (`sql`) and comes back with the match, ready for the LLM.
Bookkeeping fields (`verified`, `owner`, `source`) help the people maintaining the workbook, not the search.

#### 6 · `capability_card` — what the assistant can't answer

```text
ROW (capability_card)                              RECORD
─────────────────────                              ──────
source:      playbook                              id:       capability_card::2
description: WHAT IT CANNOT ANSWER —      ──►      EMBED:    "WHAT IT CANNOT ANSWER — orders after
             orders after 2024-12-28                          2024-12-28 (shipments run to 2024-12-30 …);
             (shipments run to 2024-12-30 …);                 web traffic, sessions or conversion …"
             web traffic, sessions …               metadata: {sheet: capability_card, source: playbook, text}
```

The description is embedded as it is, so "website visitors per day" can find "no visit data". There are only 3 of
these records and every question needs the limits — so chapter 9 also **always includes** them, search or not.

### The design for all six tabs

| Tab | 1 Embed (what questions look like) | 2 Id | 3 Metadata (filter · hand back) | 4 Skip |
|---|---|---|---|---|
| `nlq_table` | the table's description | `table::ecom.orders` | sheet, table · text | — |
| `nlq_column` | column name, type, description, synonyms, allowed values | `column::ecom.orders.total_amount` | sheet, table, column, data_type · text | **8 rows** with `embed_exempt = Y`: map coordinates (`lat`, `lon`) |
| `nlq_join` | "Join A to B on key (cardinality). description" | `join::ecom.orders->ecom.customers::customer_id` | sheet, tables (both) · text | — |
| `glossary` | term, category, meaning, **synonyms**, columns | `glossary::Revenue` | sheet, term, category, maps_to · text | — |
| `nlq_examples` | the **question** + its paraphrases — **not the SQL** | `nlq_examples::12` | sheet, tables_used, pattern · text, **sql** | — |
| `capability_card` | the description | `capability_card::2` | sheet, source · text | — |

Two decisions to discuss in class:

- **Why is the SQL in metadata, not embedded?** Users ask questions, not SQL. We embed the *question* so a user's
  question finds it; the SQL rides along in metadata and comes back with the match — ready to hand to the LLM.
- **Why a stable id?** The workbook changes. When a row is updated, an upsert with the **same id** replaces the old
  record. A random id would leave a duplicate behind (chapter 10).

Two Pinecone rules the code respects: metadata values must be text, numbers, true/false or lists of text — never
empty (`None`); and each record's metadata must stay under 40 KB (our largest is about 2.6 KB).

In [ ]:
# ─────────────────────────────────────────────────────────────
# The design as code: one small function per tab — row ──► (id, text to embed, metadata)
#
#   ROW ──► builder for its tab ──► record {id, text, metadata}
#   clean() drops empty values (Pinecone rejects None) and turns "a; b" lists into ["a", "b"]
#
# Does NOT embed or upload. Chapter 6 runs these over every row; chapter 7 uploads.
# ─────────────────────────────────────────────────────────────
def as_list(cell):
    return [part.strip() for part in str(cell).split(";") if part.strip()] if cell else []


def clean(metadata):
    return {key: value for key, value in metadata.items() if value not in (None, "", [])}


def table_record(row, number):
    text = row["description"]
    return f"table::{row['table']}", text, {"sheet": "nlq_table", "table": row["table"], "text": text}


def column_record(row, number):
    text = f"Column {row['table']}.{row['column']} ({row['data_type']}). {row['description']}"
    if row["analyst_synonyms"]:
        text += f" Also called: {row['analyst_synonyms']}."
    if row["enum_values"] and str(row["enum_values"]) != "None":
        text += f" Values: {row['enum_values']}."
    return (f"column::{row['table']}.{row['column']}", text,
            {"sheet": "nlq_column", "table": row["table"], "column": row["column"],
             "data_type": row["data_type"], "text": text})


def join_record(row, number):
    text = (f"Join {row['left_table']} to {row['right_table']} on {row['join_on']} "
            f"({row['cardinality']}). {row['description']}")
    return (f"join::{row['left_table']}->{row['right_table']}::{row['join_on']}", text,
            {"sheet": "nlq_join", "tables": [row["left_table"], row["right_table"]], "text": text})


def glossary_record(row, number):
    text = (f"{row['term']} ({row['category']}): {row['meaning']} "
            f"Also called: {row['synonyms']}. Columns: {row['maps_to']}")
    return (f"glossary::{row['term']}", text,
            {"sheet": "glossary", "term": row["term"], "category": row["category"],
             "maps_to": as_list(row["maps_to"]), "text": text})


def example_record(row, number):
    text = f"{row['question']} Also asked as: {row['paraphrases']}"                 # the question, NOT the SQL
    return (f"nlq_examples::{number}", text,
            {"sheet": "nlq_examples", "tables_used": as_list(row["tables_used"]), "pattern": row["pattern"],
             "sql": row["query"], "text": text})


def capability_record(row, number):
    text = row["description"]
    return f"capability_card::{number}", text, {"sheet": "capability_card", "source": row["source"], "text": text}


BUILDERS = {"nlq_table": table_record, "nlq_column": column_record, "nlq_join": join_record,
            "glossary": glossary_record, "nlq_examples": example_record, "capability_card": capability_record}

# The six walk-throughs from chapter 5, built by the code — compare each with the text above
WALK_THROUGHS = [("nlq_table", "table", "ecom.orders"),
                 ("nlq_column", "column", "total_amount"),
                 ("nlq_join", "left_table", "ecom.orders"),
                 ("glossary", "term", "Revenue"),
                 ("nlq_examples", "question", "What is total merchandise revenue by month?"),
                 ("capability_card", "description", "WHAT IT CANNOT ANSWER")]

for tab, key, value in WALK_THROUGHS:
    number, row = next((i, r) for i, r in enumerate(read_sheet(tab), start=1) if str(r[key]).startswith(value))
    record_id, text, metadata = BUILDERS[tab](row, number)
    print(f"── {tab}")
    print("id      :", record_id)
    print("embed   :", text[:160], "…")
    print("metadata:", {k: (v[:40] + "…" if isinstance(v, str) and len(v) > 40 else v) for k, v in clean(metadata).items()})
    print()

## Chapter 6 — Build every record

Run each tab's builder over every row. Three rows-level rules decide what is loaded:

| Rule | Column in the workbook | Today |
|---|---|---|
| Skip rows with no searchable meaning | `embed_exempt = Y` | 8 coordinate columns |
| Skip rows switched off | `active = False` | none |
| Skip rows marked for removal | `action = DELETE` | none — every row is an INSERT |

All three are checked even when they match nothing today: the workbook will change (chapter 10).

In [ ]:
# ─────────────────────────────────────────────────────────────
# Build all records
#
#   for each tab ──► for each row ──► skip? (exempt · inactive · DELETE) ──► builder ──► record
#
# Does NOT embed. Records hold text; chapter 7 turns the text into numbers.
# ─────────────────────────────────────────────────────────────
def is_yes(value):
    return str(value).strip().lower() in ("y", "yes", "true")


def skip(row):
    return (is_yes(row.get("embed_exempt"))
            or str(row.get("active")).strip().lower() == "false"
            or str(row.get("action")).strip().upper() == "DELETE")


records = []
for tab in KB_TABS:
    kept = skipped = 0
    for number, row in enumerate(read_sheet(tab), start=1):
        if skip(row):
            skipped += 1
            continue
        record_id, text, metadata = BUILDERS[tab](row, number)
        records.append({"id": record_id, "text": text, "metadata": clean(metadata)})
        kept += 1
    print(f"{tab:16} kept {kept:>3}   skipped {skipped}")

print(f"\nTOTAL: {len(records)} records")
assert len({r["id"] for r in records}) == len(records), "two records share an id"

## Chapter 7 — Load into Pinecone

Embed the text and upsert — **100 records per call**, far faster than 337 separate calls.

```text
records[0:100]    ──embed──► 100 × 1536 numbers ──upsert──► namespace "ecom-kb"
records[100:200]  ──embed──► …
records[200:300]  ──embed──► …
records[300:337]  ──embed──► …
```

Same index as Part 0, new namespace: `help-articles` (Part 0) and `ecom-kb` (this notebook) never mix.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Embed + upsert in batches of 100, then wait until all are searchable
#
#   100 texts ──► one embed call ──► 100 × {id, values, metadata} ──► upsert ──► next 100
#
# Does NOT delete anything. Running it again replaces the same ids — no duplicates.
# ─────────────────────────────────────────────────────────────
BATCH = 100

for start in range(0, len(records), BATCH):
    batch = records[start:start + BATCH]
    vectors = embed([r["text"] for r in batch])
    index.upsert(vectors=[{"id": r["id"], "values": v, "metadata": r["metadata"]} for r, v in zip(batch, vectors)],
                 namespace=NAMESPACE)
    print(f"upserted {start + 1}–{start + len(batch)}")

for _ in range(30):                                     # new records take a few seconds to become searchable
    found = index.describe_index_stats().namespaces.get(NAMESPACE)
    if found and found.vector_count >= len(records):
        break
    time.sleep(2)
print(f"\n{NAMESPACE}: {found.vector_count if found else 0} records stored")

## Chapter 8 — Retrieve: the agent's three lookups

First, the manager's question against the **whole** knowledge base:

In [ ]:
# ─────────────────────────────────────────────────────────────
# lookup(): question ──► embed ──► Pinecone top-K, optionally only some tabs ──► matches
#
# Does NOT print the SQL or full text — show() prints a short view; the matches keep everything.
# ─────────────────────────────────────────────────────────────
def lookup(question, sheets=None, top_k=5):
    result = index.query(vector=embed([question])[0], top_k=top_k, namespace=NAMESPACE, include_metadata=True,
                         filter={"sheet": {"$in": sheets}} if sheets else None)
    return result.matches


def show(title, matches):
    print(title)
    for m in matches:
        print(f"   {m.score:.3f}  {m.id:52} {m.metadata['text'][:60]}")
    print()


QUESTION = "What was our GMV last month?"
show(f"WHOLE KNOWLEDGE BASE — {QUESTION}", lookup(QUESTION, top_k=8))

Look for `glossary::Revenue` — reachable only through the word GMV in its synonyms. Notice what else comes back:
a search over everything returns a **mix** — some useful facts, some noise (columns with "last" in their name, for
"last month"?), all competing for the same top-K slots.

So the agent doesn't search once. It runs **three lookups**, each limited to the tabs it needs — the filter from
Part 0, applied to the `sheet` metadata:

| Agent tool (Version 3) | Searches tabs | Brings back |
|---|---|---|
| `lookup_schema` | `nlq_table`, `nlq_column`, `nlq_join` | where the data lives |
| `lookup_examples` | `nlq_examples` | proven questions + their SQL |
| `lookup_common` | `glossary`, `capability_card` | what words mean here, and the limits |

In [ ]:
# ─────────────────────────────────────────────────────────────
# The agent's three lookups, for one question
#
#   question ──► schema (tables · columns · joins) ──► examples ──► common (glossary · capability card)
#
# Does NOT decide which facts to use. Chapter 9 puts them together for the LLM.
# ─────────────────────────────────────────────────────────────
LOOKUPS = {
    "lookup_schema":   ["nlq_table", "nlq_column", "nlq_join"],
    "lookup_examples": ["nlq_examples"],
    "lookup_common":   ["glossary", "capability_card"],
}

retrieved = {}
for tool, sheets in LOOKUPS.items():
    retrieved[tool] = lookup(QUESTION, sheets=sheets, top_k=3)
    show(f"{tool}  {sheets}", retrieved[tool])

## Chapter 9 — The context pack: what the LLM receives ★

This is the point of the whole notebook. Instead of 42,000 tokens of everything, the LLM gets a small pack. A fact
gets into the pack in one of three ways:

```text
1 RETRIEVED   the three lookups (chapter 8)            glossary terms · schema matches · closest example + its SQL
2 FOLLOWED    metadata links from a retrieved record    glossary::Revenue → maps_to → fetch column::ecom.orders.total_amount
3 ALWAYS IN   small knowledge every question needs      the 3 capability-card records — the limits ("data ends 2024-12-28")
```

**Followed:** a search can miss the right column — but once it finds `glossary::Revenue`, the record's `maps_to`
metadata names the exact columns. We fetch those records **by id**: no search, no guessing. Metadata isn't only for
filters — it links records together.

**Always in:** the capability card is 3 short records (about 470 tokens) that matter for *every* question. Small,
always-relevant knowledge isn't left to a search — it's always included. Retrieve what is large; include what is
small and always needed.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Build the context pack
#
#   STEP 1 retrieved : glossary matches · schema matches · closest example (+ its SQL from metadata)
#   STEP 2 followed  : glossary maps_to ──► fetch those column records by id
#   STEP 3 always in : fetch the 3 capability-card records by id
#   ──► one block of text ──► size compared with the whole workbook
#
# Does NOT call an LLM. This text is what Agent Version 3 sends along with the question.
# ─────────────────────────────────────────────────────────────
ALWAYS_IN = [f"capability_card::{n}" for n in (1, 2, 3)]


def fetch_texts(ids):
    found = index.fetch(ids=ids, namespace=NAMESPACE).vectors
    return [found[i].metadata["text"] for i in ids if i in found]


def context_pack(question, found):
    terms = [m for m in found["lookup_common"] if m.metadata["sheet"] == "glossary"]            # STEP 1
    linked = sorted({column for m in terms for column in m.metadata.get("maps_to", [])})         # STEP 2
    best = found["lookup_examples"][0].metadata

    lines = [f"QUESTION: {question}", "", "WHAT THE WORDS MEAN HERE:"]
    lines += [f"- {m.metadata['text'][:400]}" for m in terms]
    lines += ["", "COLUMNS NAMED BY THOSE TERMS (fetched by id):"]
    lines += [f"- {t[:300]}" for t in fetch_texts([f"column::{c}" for c in linked])]
    lines += ["", "OTHER SCHEMA MATCHES:"]
    lines += [f"- {m.metadata['text'][:300]}" for m in found["lookup_schema"]]
    lines += ["", "A PROVEN EXAMPLE TO FOLLOW:", f"- {best['text']}", best.get("sql", "")]
    lines += ["", "LIMITS AND RULES (always included):"]                                         # STEP 3
    lines += [f"- {t}" for t in fetch_texts(ALWAYS_IN)]
    return "\n".join(lines)


pack = context_pack(QUESTION, retrieved)
print(pack)
print(f"\n── context pack ≈ {len(pack) // 4:,} tokens   (vs ≈ {total_chars // 4:,} for the whole workbook)")

**Read the pack as the LLM would.** Does it now have what chapter 2 said was missing?

| Chapter 2 needed | Look for it in the pack | Got there by |
|---|---|---|
| what GMV means here | Revenue = merchandise only, shipping and tax not included | retrieved (glossary synonyms) |
| the column and table | `ecom.orders.total_amount` | **followed** (Revenue's `maps_to`) |
| a proven query | "What is total merchandise revenue by month?" + its SQL | retrieved (example + metadata) |
| whether "last month" has data | orders end 2024-12-28 | **always in** (capability card) |

Notice the chain: the column and the limits are there **even if the search ranked them low** — because one
retrieved record (Revenue) links to them, and the limits are always included. If the glossary term itself is missing,
that's a **retrieval** problem — fix the knowledge text (chapter 5) or the lookups, not the LLM. Part 3 measures how
often retrieval gets it right.

Try it with other questions:

In [ ]:
for q in ["Which customers send back the most products?",
          "How long do deliveries take?",
          "Which coupon brings the most money?"]:
    found = {tool: lookup(q, sheets=sheets, top_k=3) for tool, sheets in LOOKUPS.items()}
    print(context_pack(q, found)[:1200], "\n…\n" + "=" * 100 + "\n")

## Chapter 10 — Keep it in sync

The database changes, so the workbook changes, so the index must follow. That is why every row has an `action`
column and every record a **stable id**:

| Workbook `action` | What we do in Pinecone | Why it works |
|---|---|---|
| `INSERT` | upsert the new record | new id → new record |
| `UPDATE` | upsert again | **same id → the old record is replaced**, no duplicate |
| `DELETE` | `index.delete(ids=[...])` | the id tells us exactly which record |

A safe demonstration — add a test record, read it back by id, replace it, delete it. Nothing real is touched.

In [ ]:
# ─────────────────────────────────────────────────────────────
# INSERT ──► fetch ──► UPDATE (same id) ──► fetch ──► DELETE, on a throw-away test record
#
# Does NOT change any of the 337 real records.
# ─────────────────────────────────────────────────────────────
def put(record_id, text):
    index.upsert(vectors=[{"id": record_id, "values": embed([text])[0],
                           "metadata": {"sheet": "test", "text": text}}], namespace=NAMESPACE)
    time.sleep(3)


def read(record_id):
    found = index.fetch(ids=[record_id], namespace=NAMESPACE).vectors
    return found[record_id].metadata["text"] if record_id in found else "(not found)"


TEST_ID = "glossary::Test Term"
put(TEST_ID, "Test Term (metric): version 1 of a made-up glossary row.")
print("INSERT →", read(TEST_ID))

put(TEST_ID, "Test Term (metric): version 2 — the meaning was corrected.")
print("UPDATE →", read(TEST_ID), "   (same id: replaced, not duplicated)")

index.delete(ids=[TEST_ID], namespace=NAMESPACE)
time.sleep(3)
print("DELETE →", read(TEST_ID))

## Chapter 11 — What vector search can't do

Ask: **"What is revenue by supplier?"** The answer needs five tables connected in order:

```text
ecom.orders ──order_id──► ecom.order_items ──product_id──► ecom.products
            ──product_id──► ecom.product_suppliers ──supplier_id──► ecom.suppliers
```

Vector search returns records that are **similar in meaning** to the question — some supplier and revenue records,
maybe one or two joins. It does **not** work out a **path**: which joins, in which order. That is structure, not
meaning.

In [ ]:
show("lookup_schema — What is revenue by supplier?", lookup("What is revenue by supplier?",
                                                            sheets=LOOKUPS["lookup_schema"], top_k=6))

You get pieces, not the path. The agent solves this differently: it reads the `nlq_join` records and **follows the
connections in code**, from the table holding revenue to the table holding suppliers. Vector search finds the
pieces; code connects them.

---

## The story in one breath

An LLM can write SQL but doesn't know **this** database → the facts it needs are in the workbook, but all of it is
~42,000 tokens per question → so we store each row as a **record** (embed what questions look like, a stable id,
metadata to filter, link and hand back) → load 337 records into Pinecone → for each question the agent runs **three
lookups**, follows the links, adds the limits → and sends the LLM a small **context pack** instead of everything → stable ids keep the index in sync →
and code, not vector search, connects tables.

## Quick knowledge check

1. Name two facts the LLM needs for "What was our GMV last month?" that it can't know by itself.
2. Why not paste the whole workbook into every prompt?
3. What are the four design decisions for each tab?
4. Why are the glossary synonyms embedded?
5. Why is an example's SQL stored in metadata instead of being embedded?
6. Why must ids be stable?
7. Which rows are skipped, and why?
8. What are the agent's three lookups, and which tabs does each search?
9. What is the context pack, and what are the three ways a fact gets into it?
10. Why can't vector search alone answer "revenue by supplier"?

The answers are in the next cell — try first.

## Answers

| # | Answer |
|---|---|
| 1 | Any two of: the table is `ecom.orders`; the column is `total_amount`; GMV here means merchandise-only Revenue; orders end 2024-12-28 |
| 2 | ~42,000 tokens per question: costly, slow, noisy — and a real company's knowledge wouldn't fit at all |
| 3 | What to embed · what id · what metadata · what to skip |
| 4 | Users use those words. "GMV" appears only in the Revenue synonyms; embedding it lets a GMV question find Revenue |
| 5 | Users ask questions, not SQL — so we embed the question; the SQL comes back with the match, ready for the LLM |
| 6 | So an UPDATE replaces the same record (no duplicate) and a DELETE knows exactly which record to remove |
| 7 | `embed_exempt = Y` (8 map coordinates — no searchable meaning), `active = False` and `action = DELETE` (none today) |
| 8 | `lookup_schema` (tables, columns, joins) · `lookup_examples` (examples) · `lookup_common` (glossary, capability card) |
| 9 | What the LLM receives instead of the whole workbook: retrieved facts (glossary, schema, closest example + SQL), columns followed through `maps_to`, and the capability card, always included |
| 10 | It returns records similar in meaning; it doesn't work out which joins connect the tables, in which order. Code follows the `nlq_join` records for that |

**Try it yourself**

1. Ask `"website visitors per day"` with `lookup_common`. Which capability-card record comes first — and what
   should the assistant answer?
2. Remove `Also called: {row['synonyms']}` from `glossary_record`, rebuild and reload. Does the GMV question still
   find `glossary::Revenue`?
3. Change `top_k` in the three lookups to 1 and to 10. How do the context pack's size and quality change?

**Next — Part 3:** measure how often retrieval finds the right records, with ground truth.